# Entrenamiento de un modelo NER personalizado con spaCy

**Curso:** Procesamiento de Lenguaje Natural · Unidad 4 — Etiquetado de secuencias y reconocimiento de entidades nombradas  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17 — Sequence Labeling for Parts of Speech and Named Entities  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook muestra cómo **entrenar un reconocedor de entidades nombradas (NER) propio con spaCy** a partir de ejemplos anotados con posiciones de caracteres: dónde empieza y termina cada entidad y de qué tipo es (organización, lugar…). Se verifica que las anotaciones estén bien alineadas con los tokens, se crea un *pipeline* vacío en español con un componente NER, se entrena con la API `Example`, y el modelo se guarda, se recarga y se prueba con una frase nueva.

**Objetivo:** aprender el procedimiento para crear un NER con **categorías propias de un negocio** (que los modelos genéricos no reconocen) y detectar los errores de anotación más comunes antes de entrenar.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Energía e industria — Equipos en órdenes de trabajo

- **Problema:** el modelo general de spaCy no reconoce equipos eléctricos.
- **Entrada:** órdenes de trabajo (`cambio de reconectador en circuito 12`).
- **Cómo se aplica el entrenamiento de NER en spaCy:**
  1. Se anotan ejemplos con posiciones de caracteres (inicio, fin, `EQUIPO`).
  2. Se verifica que cada anotación coincida con límites de tokens antes de entrenar.
  3. Se entrena con la API `Example`, se guarda el modelo y se prueba con órdenes nuevas.
- **Salida:** equipos detectados en cada orden.
- **Decisión:** estadísticas de fallas por tipo de equipo.

### Retail, agro y puertos — Productos en pedidos por chat

- **Problema:** una ferretería recibe pedidos por WhatsApp en lenguaje libre.
- **Entrada:** `mándame 3 bultos de cemento Argos`.
- **Cómo se aplica el entrenamiento de NER en spaCy:**
  1. Se anotan PRODUCTO, CANTIDAD y MARCA por caracteres.
  2. Se verifica la alineación de las anotaciones con los tokens.
  3. Se entrena el NER propio y se evalúa con mensajes no vistos.
- **Salida:** líneas del pedido (producto, cantidad, marca).
- **Decisión:** pedido prellenado que el vendedor confirma.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install spacy
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `json, random, pathlib.Path` | Lectura de anotaciones, barajado y rutas |
| `spacy` | Pipeline en blanco, componente NER, entrenamiento, guardado y carga |
| `spacy.training.Example, offsets_to_biluo_tags` | Ejemplos de entrenamiento y verificación de la alineación |

**Datos:** `data/datasetNER.json` (incluido en el repositorio; contiene solo 2 oraciones de ejemplo).

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*; Studer et al., 2021). El proceso tiene seis fases, y en cada una se identifican los riesgos de calidad y cómo se mitigan. Las fases aparecen marcadas con 🔷 a lo largo del notebook.

| Fase CRISP-ML(Q) | Dónde se aborda en este notebook | Aseguramiento de calidad |
|------------------|----------------------------------|--------------------------|
| 1. Entendimiento del negocio y de los datos | Secciones 2 y 4: formato de las anotaciones, carga y **verificación de la alineación** | Toda entidad desalineada se reporta antes de entrenar |
| 2. Ingeniería de datos | Sección 5: registro de etiquetas e inicialización | Solo se registran etiquetas presentes en los datos |
| 3. Ingeniería del modelo | Secciones 3 y 6: pipeline en blanco y bucle de entrenamiento | Semilla fija y datos barajados en cada época |
| 4. Aseguramiento de calidad del modelo | Sección 7: prueba en una frase nueva y puntajes sobre los datos de entrenamiento | Se advierte que con 2 ejemplos no hay evaluación real posible |
| 5. Despliegue | Sección 7: guardado con `to_disk` y carga con `spacy.load` | Se verifica que el modelo recargado funcione |
| 6. Monitoreo y mantenimiento | Al final (fuera de alcance) | — |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | La verificación de alineación (sección 4.1) encontró un error real: la entidad *California* estaba anotada con un desfase de un carácter y spaCy la ignoraba al entrenar. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Se explica qué hace cada etapa del entrenamiento y cómo se representa cada entidad (BILUO). |
| **ISO/IEC 42001:2023** — Sistema de gestión de IA en una organización | Se documenta que el conjunto de datos de ejemplo es mínimo y no permite medir la calidad del modelo. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | spaCy es de código abierto (MIT) y el modelo entrenado se puede compartir con sus datos de entrenamiento. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Dar formato a datos anotados para entrenar NER en spaCy (entidades como posiciones de caracteres)
2. **Verificar** que cada anotación esté alineada con los límites de los tokens
3. Crear un *pipeline* vacío de spaCy y agregar un componente **NER**
4. Entrenar el modelo con ejemplos supervisados y la API `Example`
5. Guardar y recargar el modelo entrenado, y hacer inferencia

## ¿Qué es el reconocimiento de entidades nombradas (NER)?

NER es una tarea de etiquetado de secuencias que identifica y clasifica **entidades nombradas** (personas, organizaciones, lugares, fechas, etc.) en un texto. El NER de spaCy es un reconocedor **basado en transiciones** sobre una red neuronal.

---

## 1. Configuración del entorno

In [ ]:
import json
import random
from pathlib import Path

import spacy
from spacy.training import Example, offsets_to_biluo_tags

DIR_DATA = Path.cwd() / 'data'
random.seed(42)   # barajado reproducible

> 🔷 **CRISP-ML(Q) · Fase 1 — Entendimiento del negocio y de los datos.** Formato de las anotaciones, carga y verificación de la alineación.

---

## 2. Verificar el formato de las posiciones de las entidades

spaCy necesita las entidades como posiciones de caracteres: `(inicio, fin, ETIQUETA)`, donde `fin` es exclusivo (como en `texto[inicio:fin]`). La función `offsets_to_biluo_tags` convierte esas posiciones en etiquetas por token para inspeccionarlas.

In [ ]:
# Comprueba que las posiciones de las entidades coincidan con los límites de los tokens
nlp_blank = spacy.blank('es')
text = 'Apple lanzó el nuevo iPhone en California.'
entities = [(0, 5, 'ORG'), (31, 41, 'LOC')]   # texto[31:41] == 'California'

for inicio, fin, etiqueta in entities:
    print(f'{etiqueta}: {text[inicio:fin]!r}')
tags = offsets_to_biluo_tags(nlp_blank.make_doc(text), entities)
print(f'\nTexto: {text}')
print(f'Etiquetas BILUO: {tags}')

> **Interpretación de la salida:** las etiquetas BILUO siguen el esquema *Begin–Inside–Last–Unit–Outside*. *Apple* y *California* son entidades de un solo token, así que reciben `U-ORG` y `U-LOC` (*Unit*). Una entidad de varios tokens usaría `B-` en el primero, `I-` en los intermedios y `L-` en el último. Un guion `-` significa que una posición **no coincide** con el límite de un token: spaCy ignora esa entidad al entrenar, así que hay que corregirla antes.

---

## 3. Crear el pipeline y agregar el componente NER

In [ ]:
# Modelo vacío en español: solo tokenizador, sin pesos preentrenados
nlp = spacy.blank('es')

# Agrega el componente NER al pipeline
ner = nlp.add_pipe('ner')
print(f'Pipeline: {nlp.pipe_names}')

---

## 4. Cargar los datos anotados

In [ ]:
# Formato: lista de {"text": str, "entities": [[inicio, fin, etiqueta], ...]}
with open(DIR_DATA / 'datasetNER.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

print(f'Ejemplos de entrenamiento: {len(train_data)}')
print(f'Primer ejemplo: {train_data[0]}')

### 4.1 Verificar la alineación de todas las anotaciones

Antes de entrenar, revisamos que **cada** entidad del archivo coincida con los límites de los tokens. Una entidad desalineada se ignora en silencio durante el entrenamiento.

In [ ]:
# Revisa cada entidad: texto anotado y etiquetas BILUO; '-' indica desalineación
problemas = 0
for i, entry in enumerate(train_data):
    doc = nlp.make_doc(entry['text'])
    tags = offsets_to_biluo_tags(doc, [tuple(e) for e in entry['entities']])
    for inicio, fin, etiqueta in entry['entities']:
        print(f'[{i}] {etiqueta:4s} {entry["text"][inicio:fin]!r}')
    if '-' in tags:
        problemas += 1
        print(f'    ⚠️ Desalineada en el ejemplo {i}: {list(zip([t.text for t in doc], tags))}')
print(f'\nEjemplos con entidades desalineadas: {problemas} de {len(train_data)}')

> 🔷 **CRISP-ML(Q) · Fase 2 — Ingeniería de datos.** Registro de etiquetas e inicialización del modelo.

---

## 5. Registrar las etiquetas e inicializar

In [ ]:
# Registra todas las etiquetas de entidad presentes en los datos
for entry in train_data:
    for _, _, label in entry['entities']:
        ner.add_label(label)

# Inicializa los pesos del modelo y devuelve el optimizador
optimizer = nlp.initialize()
print(f'Etiquetas NER: {ner.labels}')

> 🔷 **CRISP-ML(Q) · Fase 3 — Ingeniería del modelo.** Bucle de entrenamiento.

---

## 6. Bucle de entrenamiento

Se entrena durante 30 épocas. Los datos se barajan en cada época para reducir el efecto del orden.

In [ ]:
n_epochs = 30

for epoch in range(n_epochs):
    random.shuffle(train_data)
    losses = {}
    for entry in train_data:
        # Ejemplo = documento tokenizado + entidades anotadas
        doc = nlp.make_doc(entry['text'])
        example = Example.from_dict(doc, {'entities': entry['entities']})
        nlp.update([example], sgd=optimizer, losses=losses)
    if epoch % 5 == 0:
        print(f'Época {epoch:2d}  pérdida NER: {losses["ner"]:.4f}')

> **Interpretación de la salida:** la pérdida del NER disminuye con las épocas: el modelo aprende a predecir los límites y tipos de las entidades anotadas. Con solo 2 oraciones, el modelo **memoriza** esos ejemplos; para generalizar se necesitan cientos de oraciones anotadas. Si la pérdida se estanca o sube, revisa la tasa de aprendizaje, agrega datos o parte de un modelo preentrenado (`es_core_news_sm`) en lugar de uno vacío.

> 🔷 **CRISP-ML(Q) · Fase 4 — Aseguramiento de calidad del modelo · Fase 5 — Despliegue.** Guardado, recarga, inferencia y puntajes.

---

## 7. Guardar y recargar el modelo

In [ ]:
# Guarda el pipeline completo en una carpeta
model_path = Path('modelo_ner_personalizado')
nlp.to_disk(model_path)
print(f'Modelo guardado en: {model_path.resolve()}')

# Recarga y prueba con una frase nueva
nlp2 = spacy.load(model_path)
test_text = 'Google abrió una nueva oficina en Madrid.'
doc = nlp2(test_text)

print(f'\nTexto de prueba: {test_text}')
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  [{ent.label_}] {ent.text!r}')

> **Interpretación de la salida:** el modelo recargado (`nlp2`) hace inferencia sobre una frase nueva y muestra cada entidad como `[ETIQUETA] texto`. La calidad depende del tamaño y la variedad de `datasetNER.json`: con 2 ejemplos es probable que falle en frases distintas. Para un uso real, evalúa con precisión, *recall* y F1 sobre un conjunto de prueba separado, por ejemplo con `nlp.evaluate()` o `spacy evaluate`.

### 7.1 Puntajes sobre los datos de entrenamiento

`nlp.evaluate()` calcula precisión, *recall* y F1 por entidad. Aquí lo aplicamos a los **mismos** ejemplos de entrenamiento solo para mostrar la API: un puntaje alto indica que el modelo memorizó, **no** que generalice.

In [ ]:
# Evaluación sobre los ejemplos de entrenamiento (solo ilustrativa: no mide generalización)
ejemplos = [Example.from_dict(nlp2.make_doc(e['text']), {'entities': e['entities']}) for e in train_data]
puntajes = nlp2.evaluate(ejemplos)
print({k: round(v, 3) for k, v in puntajes.items() if k in ('ents_p', 'ents_r', 'ents_f')})

---

> 🔷 **CRISP-ML(Q) · Fase 6 — Monitoreo y mantenimiento** (fuera de alcance): registrar las entidades detectadas por tipo y revisar periódicamente una muestra. Cada error corregido a mano es una nueva anotación para reentrenar.